In [ ]:
import pandas as pd
import numpy as np
import gspread
from pythonbq import pythonbq
from oauth2client.service_account import ServiceAccountCredentials
from redash_dynamic_query import RedashDynamicQuery
import pymysql     
from datetime import datetime
from datetime import date
import time
from sqlalchemy import create_engine
from access.keys import spread_sheet

################################################## FUNCTIONS ################################################

from access.keys import bigquery_project

myProject = pythonbq(
    bq_key_path=bigquery_project['key_path'],
    project_id=bigquery_project['id']
)

###########################################* MYSQL FUNCTIONS *################################################

import pymysql
pymysql.install_as_MySQLdb()
from access.data_team_creds import mysql_master_aws, mysql_replica_aws

read_engine = create_engine(mysql_replica_aws['uri'])
write_engine = create_engine(mysql_master_aws['uri'])

con = pymysql.connect(host=mysql_master_aws['host'], port = mysql_master_aws['port'], user=mysql_master_aws['user'], password=mysql_master_aws['password'])

cursorObject = con.cursor()

#############################################################################################################

print("Connection Built : Done")

import time
time.sleep(30)

#############################################################################################################

print("Starting the Job")
################################google sheet read ########################################################

def clean_google_sheet(worksheet, headers):
    worksheet.clear()
    worksheet.insert_row(headers, index=1)

######################################### FUNCTIONS TO READ G SHEET DATA IN PYTHON ############################################

scope = ['https://spreadsheets.google.com/feeds','https://www.googleapis.com/auth/drive'] 

json_cred = spread_sheet['key_path']

creds = ServiceAccountCredentials.from_json_keyfile_name(json_cred, scope)
client = gspread.authorize(creds)


sheet = client.open_by_url('https://docs.google.com/spreadsheets/d/1qUHh7zkXQ-T6Ru7xvpgzX5QqDxPtRhOdWPhBBwFHS/edit?gid=0#gid=0') ##upload my worksheet
worksheet = sheet.get_worksheet(0) ## Reading sheet 0 - raw data.

data = worksheet.get_all_records()

if len(data) > 0:
    df = pd.DataFrame(data)
    df.columns = ["ib_lead_id", "campaign"]
    # df = df[['dcs_id']] ##lic_id
    df['ib_lead_id'] = pd.to_numeric(df['ib_lead_id'],errors='coerce').fillna(0).astype('int64')
    df["campaign"] = df["campaign"].astype(str)

    df = df[df['ib_lead_id']>0]
    print(df.info())

    df.to_gbq('vyapar_data.ib_lead_for_target_calling', if_exists='append',  project_id='vyapar-faf62')  ##

    df['inserted_at']=datetime.now()
    df.to_gbq('vyapar_data.dummy_ib_lead_for_target_calling', if_exists='append',  project_id='vyapar-faf62')

# Clear all data except the header row (row 1)
clean_google_sheet(worksheet, ['ib_lead_id','campaign'])  ##dcs_id 

print("Data Pushed For Target Calling.")

##################################################################


Query_app_opened = """
with ib_lead as
(
  select * from 
  (
select id,source_id,phone,calling_code,platform
,alternate_phone,email,business_name,alternate_email, business_type, name, city, first_desktop_login_date, first_mobile_login_date, device_id,call_disposition,call_comments,assigned_dt,alternate_platform_device_id,lead_status
,agent_id,identity,created_at ,language
,business_category,current_software,follow_up_date,source
,country
,case when call_disposition in 
(
'WN:invalid_number',
'NI:dnd',
'WN:vyp_employee',
'WN:not_exist',
'invalid_number',
'Converted'
) or lead_status in ('converted','bought','alternate_bought') then 'no' else 'yes' end valid_to_call
from `vyapar_mysql.vyapar_ib_leads` 
where source<>'PARTNER_LEADS'
and converted_at is null
and (date(follow_up_date)<current_date or follow_up_date is null)
and agent_id is not null
and agent_id not in (14328308,7550789,14508603,14471333)
and (is_closed<>1 or is_closed is null)

-- 14328308	ib_ur_dummy@vyaparapp.in
-- 7550789	917888940970@vyaparapp.in (invalid dummy)
-- 14508603	ib_mobile_dummy@vyaparapp.in
-- 14471333	indian_and_non_gulf_ibleads_dummy@vyaparapp.in
  ) where valid_to_call='yes'
)
,plans_and_pricing_accessed_count AS (
  SELECT distinct LTRIM(REGEXP_REPLACE(verified_contact, '[^0-9]', ''), '0')  verified_contact, 
  device_id, time AS event_date,platform
  FROM vyapar-faf62.mixpanel_export.plans_and_pricing_accessed
  WHERE DATE(_PARTITIONTIME)=current_date           --- events today
)
,buy_now_clicked_cnt AS (
  SELECT distinct LTRIM(REGEXP_REPLACE(verified_contact, '[^0-9]', ''), '0')  verified_contact, 
  device_id, time AS event_date,platform
  FROM vyapar-faf62.mixpanel_export.buy_now_clicked
  WHERE DATE(_PARTITIONTIME)=current_date         --- events today
)

, base_data as 
(
  select *except(rnk) 
  from (
  --App_open
  select a.* 
  ,case when dao.device_id is not null then 'Active_on_App' end call_type
  ,active_on_app
  ,6 rnk
  from ib_lead a 
join 
(SELECT distinct macAddress device_id
, updated_at active_on_app,plan_name,license_code
FROM `vyapar-faf62.vyapar_mysql.vyapar_desktop_usage_details` 
where macAddress is not null and timestamp(updated_at) >= timestamp(FORMAT_DATETIME('%Y-%m-%d 10:00:00', current_datetime)  ) 
) dao on a.device_id=dao.device_id

--Desktop_Login_Today
union all 
select a.* 
  ,'Desktop_Login_Today' call_type
  ,first_desktop_login_date active_on_app
  ,5 rnk
  from ib_lead a where date(first_desktop_login_date)=current_date and (platform=1 or platform is null)


--Target_camp
union all 
select a.* 
  ,concat('Target_camp: ' ,campaign) lead_type
  ,current_timestamp() + interval 330 minute event_time
  ,10 rnk
from ib_lead a 
join (select ib_lead_id,max(campaign) campaign from  `vyapar-faf62.vyapar_data.ib_lead_for_target_calling` group by 1) b on a.id=b.ib_lead_id

--Failed_payment
union all 
select a.* 
  ,case when fp.phone is not null then 'Failed_Pament' end lead_type
  ,event_date
  ,1 rnk
from ib_lead a 
join 
(select LTRIM(REGEXP_REPLACE(phone, '[^0-9]', ''), '0') phone,max(latest_payment_tried_time) event_date
from vyapar_mysql.vyapar_data_mobilefailed_payments_new
where date(latest_payment_tried_time)=current_date
and LTRIM(REGEXP_REPLACE(phone, '[^0-9]', ''), '0') is not null
group by 1
) fp on a.phone=fp.phone

-- buy_now_clicked
union all 
select a.* 
  ,case when event_date is not null then 'buy_now_clicked' end lead_type
  ,event_date
  ,2 rnk
  from ib_lead a 
join 
(SELECT verified_contact,max(event_date) event_date
FROM `buy_now_clicked_cnt` 
where length(verified_contact)>3
group by 1
 ) pa on a.phone=pa.verified_contact

 union all 
select a.* 
  ,case when event_date is not null then 'buy_now_clicked' end lead_type
  ,event_date
  ,2 as rnk
  from ib_lead a 
join 
(SELECT device_id,max(event_date) event_date
FROM `buy_now_clicked_cnt` 
where device_id is not null
group by 1
 ) pa on a.device_id=pa.device_id


-- plans_and_pricing_accessed
union all 
select a.* 
  ,case when event_date is not null then 'plans_and_pricing_clicked' end lead_type
  ,event_date
  ,3 rnk
  from ib_lead a 
join 
(SELECT verified_contact,max(event_date) event_date
FROM `plans_and_pricing_accessed_count` 
where length(verified_contact)>3
group by 1
 ) pa on a.phone=pa.verified_contact

 union all 
select a.* 
  ,case when event_date is not null then 'plans_and_pricing_clicked' end lead_type
  ,event_date
  ,3 as rnk
  from ib_lead a 
join 
(SELECT device_id,max(event_date) event_date
FROM `plans_and_pricing_accessed_count` 
where device_id is not null
group by 1
 ) pa on a.device_id=pa.device_id



union all 
select a.*
,messageType lead_type
,createdat event_time
,5 as rnk
from ib_lead a
join (
SELECT `from` response_number,TIMESTAMP(max(createdat)) createdat,max(messageType) messageType
FROM `vyapar-faf62.mongo_communiverse.whatsappquickreplydetails` 
WHERE date(_v_partitionDate) >= date_trunc(current_date(),month)
and messageType in ('GULF_SALE_2','GULF_SALE_INTENT')
and date(createdAt)=current_date()
group by 1) wa on wa.response_number=a.identity

) qualify row_number() over(partition by id order by rnk,active_on_app desc)=1
)



,app_open as 
(select * from 
(
select *
,0 primary_priority_key 
,ROW_NUMBER() OVER(PARTITION BY agent_ID ORDER BY active_on_app) secondry_priority_key
from BASE_DATA
) 
where  
id not in 
(select distinct id_team, 
from vyapar_data.data_autodialer_daily_leads
where team = 30 AND 
(JSON_EXTRACT_SCALAR(json_string,'$.call_type')
in ('Active_on_App','buy_now_clicked','plans_and_pricing_clicked','Failed_Pament','Desktop_Login_Today','GULF_SALE_2','GULF_SALE_INTENT')
-- or JSON_EXTRACT_SCALAR(json_string,'$.call_type') like '%access_locked_on%'
or JSON_EXTRACT_SCALAR(json_string,'$.call_type') like '%Target_camp%'
)
and primary_priority_sequence = 0
)

-- Leads should not have connected (Just Picked) call today.
and id not in 
(select distinct t1.source_id
from `vyapar_mysql.vyapar_outcalls_logs` t1
where t1.source_type='ib_sales_autodialer' 
and t1.status = '1' and ifnull(cast(t1.duration as int64),0) >= 0
and date(starttime) = current_date()
)
)

,app_open_2 as 
 (
select a.* 
  ,case when call_type='Failed_Pament' then 'Failed_Pament_Redial' when call_type='Desktop_Login_Today' then 'Desktop_Login_Today_Redial' end call_type
  ,first_desktop_login_date active_on_app
,case when call_type='Failed_Pament' then -10 when call_type='Desktop_Login_Today' then -20 end primary_priority_key 
,ROW_NUMBER() OVER(PARTITION BY agent_ID ORDER BY assigned_dt) secondry_priority_key
  from ib_lead  a 
  join 
(select distinct source_id,call_type
from 
(
SELECT source_id,call_type,count(distinct __db_uuid) total_call
,timestamp(max(starttime)) last_call_time
,count(distinct case when status='1' then __db_uuid end) con_calls
FROM `vyapar-faf62.mongo_dumpdb.autodialer_call_id`	a
join 	`vyapar_mysql.vyapar_outcalls_logs` t1 on a.call_id=t1.__db_uuid
where t1.source_type='ib_sales_autodialer' 
and call_type in ('Failed_Pament','Desktop_Login_Today')
and date(starttime)=current_date
group by 1,2
) 
WHERE ifnull(con_calls,0)=0	and ifnull(total_call,0)=1
and current_timestamp() + interval 330 minute>last_call_time + interval 180 minute
) b on a.id=b.source_id
where a.id not in 
(select distinct id_team, 
from vyapar_data.data_autodialer_daily_leads
where team = 30 AND 
JSON_EXTRACT_SCALAR(json_string,'$.call_type')
in ('Desktop_Login_Today_Redial','Failed_Pament_Redial')
)

)

select ID id_team,30 team ,agent_id
,primary_priority_key as primary_priority_sequence
,secondry_priority_key as secondary_priority_sequence
,TO_JSON_STRING(STRUCT(id,
email,
alternate_email,
phone,
calling_code, 
identity,
platform,
first_mobile_login_date,
first_desktop_login_date,
device_id,
alternate_platform_device_id,
agent_id,
lead_status,
business_name,
business_type,
business_category,
current_software,
language,
alternate_phone,
assigned_dt,
follow_up_date,
source,
call_disposition,
country,
call_type
)) as json_string
,0 as called_status
from 
(
select * from app_open
union all 
select * from app_open_2
)

"""

app_opened_df = myProject.query(sql = Query_app_opened)

print(app_opened_df.groupby(['primary_priority_sequence', 'secondary_priority_sequence']).id_team.nunique().reset_index())

app_opened_df['primary_priority_sequence'] = app_opened_df['primary_priority_sequence'].astype(int)
app_opened_df['secondary_priority_sequence'] = app_opened_df['secondary_priority_sequence'].astype(float)
app_opened_df.info()

print("Pushing 15 mins leads to main table")

app_opened_df.to_sql(schema="vyapar_data", name="data_autodialer_daily_leads", con=write_engine, if_exists = 'append', index=False)

app_opened_df.to_gbq('vyapar_data.data_autodialer_daily_leads', if_exists='append',  project_id='vyapar-faf62')

print("IPT App Opened Leads Data with higher priority pushed into table data_autodialer_daily_leads")

# BQ logs
app_opened_df['created_at'] = datetime.today().strftime('%Y-%m-%d %H:%M:%S')
app_opened_df.to_gbq('vyapar_data.data_autodialer_daily_leads_logs', if_exists='append',  project_id='vyapar-faf62')

print("Data Pushed to final tables.")

print("write an update for sql table to marked lesser priority leads as called in case of duplicate leads with higher priority.")


update_leads = """select t1.id update_id
                    from vyapar_data.data_autodialer_daily_leads t1
                    JOIN vyapar_data.data_autodialer_daily_leads t2
                    ON t1.id_team = t2.id_team AND t1.team = t2.team AND t1.primary_priority_sequence > t2.primary_priority_sequence
        
                    where t1.called_status = 0 and 
                    t1.team in (30)  ; 
                """

result = pd.read_sql(update_leads, write_engine)


if len(result) > 0:
    print("length is greater then 0")
    update_id = result.iloc[:,0].to_list()

    status_called_status = """ 
    UPDATE vyapar_data.data_autodialer_daily_leads 
    SET called_status = 5
    where id in ({0});
    """.format(','.join(map(str, update_id)))

    print(status_called_status)
    cursorObject.execute(status_called_status)
    con.commit()
    # cursorObject.close()


 ################ Delete target calling table data #######
delete_data_bq = """ DELETE FROM vyapar_data.ib_lead_for_target_calling where 1=1 """
myProject.query(sql=delete_data_bq)
print("target calling table deleted successfully")

print("Job Done")   